# Breast Cancer Detection: End-to-End ML Project

This notebook builds an end-to-end pipeline for predicting malignant (1) vs benign (0) tumors using the Breast Cancer Wisconsin (Diagnostic) Dataset.

- Data source: UCI Machine Learning Repository (wdbc.data) provided in the project folder
- Models: Logistic Regression, Random Forest, SVM (optionally XGBoost if available)
- Evaluation: Accuracy, Precision, Recall, F1, ROC-AUC, Confusion Matrix
- Interpretability: Feature importance (tree-based), SHAP values
- Outputs: Figures to `reports/figures/`, best model to `models/`, metrics table to `reports/`


In [ ]:
# Setup: imports, paths, and directories
import os
import sys
import warnings
warnings.filterwarnings('ignore')

# Data and output paths
DATA_DIR = os.path.join('breast+cancer+wisconsin+diagnostic')
DATA_FILE = os.path.join(DATA_DIR, 'wdbc.data')
REPORTS_DIR = os.path.join('reports')
FIGURES_DIR = os.path.join(REPORTS_DIR, 'figures')
MODELS_DIR = os.path.join('models')

# Ensure directories exist
for d in [REPORTS_DIR, FIGURES_DIR, MODELS_DIR]:
    os.makedirs(d, exist_ok=True)

# Utility: safe import for optional libs
try:
    import shap  # type: ignore
    HAS_SHAP = True
except Exception:
    HAS_SHAP = False

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, classification_report
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

# Optional: XGBoost
try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except Exception:
    HAS_XGB = False

print('Paths ready. SHAP:', HAS_SHAP, '| XGBoost:', HAS_XGB)



In [ ]:
# 1) Data Acquisition & Loading
# UCI column names for WDBC dataset
from pathlib import Path

feature_names = [
    'radius_mean','texture_mean','perimeter_mean','area_mean','smoothness_mean',
    'compactness_mean','concavity_mean','concave_points_mean','symmetry_mean','fractal_dimension_mean',
    'radius_se','texture_se','perimeter_se','area_se','smoothness_se',
    'compactness_se','concavity_se','concave_points_se','symmetry_se','fractal_dimension_se',
    'radius_worst','texture_worst','perimeter_worst','area_worst','smoothness_worst',
    'compactness_worst','concavity_worst','concave_points_worst','symmetry_worst','fractal_dimension_worst'
]
columns = ['id','diagnosis'] + feature_names

# Resolve absolute path and fallbacks
project_root = Path.cwd()
possible_paths = [
    project_root / DATA_FILE,
    project_root / 'breast+cancer+wisconsin+diagnostic' / 'wdbc.data',
    Path(__file__).resolve().parents[1] / 'breast+cancer+wisconsin+diagnostic' / 'wdbc.data' if '__file__' in globals() else None
]
possible_paths = [p for p in possible_paths if p is not None]
existing = next((p for p in possible_paths if p.exists()), None)

if existing is not None:
    df = pd.read_csv(existing, header=None, names=columns)
    # Map target: M -> 1 (malignant), B -> 0 (benign)
    df['target'] = df['diagnosis'].map({'M': 1, 'B': 0})
    print('Loaded local CSV:', existing)
else:
    # Fallback to sklearn dataset
    print('Local CSV not found. Falling back to sklearn.datasets.load_breast_cancer')
    from sklearn.datasets import load_breast_cancer
    data = load_breast_cancer()
    sk_cols = list(data.feature_names)
    sk_df = pd.DataFrame(data.data, columns=sk_cols)
    # Map sklearn names (e.g., 'mean radius') -> UCI names (e.g., 'radius_mean')
    mapping = {
        'mean radius':'radius_mean','mean texture':'texture_mean','mean perimeter':'perimeter_mean','mean area':'area_mean','mean smoothness':'smoothness_mean',
        'mean compactness':'compactness_mean','mean concavity':'concavity_mean','mean concave points':'concave_points_mean','mean symmetry':'symmetry_mean','mean fractal dimension':'fractal_dimension_mean',
        'radius error':'radius_se','texture error':'texture_se','perimeter error':'perimeter_se','area error':'area_se','smoothness error':'smoothness_se',
        'compactness error':'compactness_se','concavity error':'concavity_se','concave points error':'concave_points_se','symmetry error':'symmetry_se','fractal dimension error':'fractal_dimension_se',
        'worst radius':'radius_worst','worst texture':'texture_worst','worst perimeter':'perimeter_worst','worst area':'area_worst','worst smoothness':'smoothness_worst',
        'worst compactness':'compactness_worst','worst concavity':'concavity_worst','worst concave points':'concave_points_worst','worst symmetry':'symmetry_worst','worst fractal dimension':'fractal_dimension_worst'
    }
    sk_df = sk_df.rename(columns=mapping)
    # Ensure column order and names match feature_names
    missing_cols = [c for c in feature_names if c not in sk_df.columns]
    if missing_cols:
        raise ValueError(f"Missing expected columns from sklearn fallback: {missing_cols}")
    df = sk_df[feature_names].copy()
    df['target'] = pd.Series(data.target, index=df.index).map({0:1, 1:0})

print('Shape:', df.shape)
df.head()


In [ ]:
# 2) Exploratory Data Analysis (EDA)
# Summary statistics
summary_stats = df[feature_names].describe().T
summary_stats.head(10)


In [ ]:
# Class distribution
class_counts = df['target'].value_counts().rename(index={0:'Benign (0)',1:'Malignant (1)'})
plt.figure(figsize=(5,4))
ax = class_counts.plot(kind='bar', color=['#69b3a2','#d95f02'])
plt.title('Class Distribution')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'class_distribution.png'), dpi=150)
plt.show()
class_counts


In [ ]:
# Correlation heatmap (top correlated features with target)
plt.figure(figsize=(10,8))
cor = df[feature_names + ['target']].corr()
sns.heatmap(cor, cmap='RdBu_r', center=0)
plt.title('Correlation Heatmap (All Features + Target)')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'correlation_heatmap.png'), dpi=150)
plt.show()

# Top features by absolute correlation with target
target_corr = cor['target'].drop('target').abs().sort_values(ascending=False)
target_corr.head(15)


In [ ]:
# Histograms for selected features
requested = ['radius_mean','texture_mean','perimeter_mean','area_mean','concavity_mean']
selected = [c for c in requested if c in df.columns]
if len(selected) == 0:
    print('No requested features found; available sample:', list(df.columns)[:5])
else:
    fig, axes = plt.subplots(nrows=1, ncols=len(selected), figsize=(16,3))
    for i, col in enumerate(selected):
        sns.histplot(df[col], bins=30, kde=True, ax=axes[i], color='#4C72B0')
        axes[i].set_title(col)
    plt.tight_layout()
    plt.savefig(os.path.join(FIGURES_DIR, 'histograms_selected.png'), dpi=150)
    plt.show()

# Boxplots to inspect outliers
if len(selected) > 0:
    plt.figure(figsize=(10,5))
    sns.boxplot(data=df[selected], orient='h')
    plt.title('Boxplots: Selected Features')
    plt.tight_layout()
    plt.savefig(os.path.join(FIGURES_DIR, 'boxplots_selected.png'), dpi=150)
    plt.show()
else:
    print('Skipping boxplots; no selected features present.')


In [ ]:
# Pairplot (sampled for speed)
sns.pairplot(df[['target'] + selected + ['texture_worst']].sample(n=min(200, len(df)), random_state=42),
             hue='target', diag_kind='kde', corner=True)
plt.suptitle('Pairplot (Sample)', y=1.02)
plt.savefig(os.path.join(FIGURES_DIR, 'pairplot_sample.png'), dpi=150)
plt.show()


In [ ]:
# 3) Data Preprocessing
# Check missing values
missing = df[feature_names + ['target']].isnull().sum().sum()
print('Total missing values:', missing)

# Features and target
X = df[feature_names].copy()
y = df['target'].astype(int)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print('Train:', X_train.shape, '| Test:', X_test.shape)



In [ ]:
# Generate visualization gallery HTML
import os
import sys
import subprocess
from pathlib import Path

project_root = Path.cwd()
script_path = project_root / 'scripts' / 'compile_visualizations.py'

if script_path.exists():
    print('Running:', script_path)
    res = subprocess.run([sys.executable, str(script_path)], capture_output=True, text=True)
    print(res.stdout)
    if res.returncode != 0:
        print(res.stderr)
    out_file = project_root / 'reports' / 'visualizations' / 'visualizations.html'
    print('Gallery:', out_file if out_file.exists() else 'Gallery not found')
else:
    print('Visualization script not found at', script_path)



In [ ]:
# 4) Model Building & Training
models = {
    'LogisticRegression': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(max_iter=500, solver='lbfgs', n_jobs=None))
    ]),
    'RandomForest': Pipeline([
        ('clf', RandomForestClassifier(n_estimators=300, random_state=42, class_weight=None))
    ]),
    'SVM': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', SVC(kernel='rbf', C=1.0, gamma='scale', probability=True, random_state=42))
    ])
}

if 'HAS_XGB' in globals() and HAS_XGB:
    models['XGBoost'] = Pipeline([
        ('clf', XGBClassifier(
            n_estimators=300,
            max_depth=4,
            learning_rate=0.05,
            subsample=0.9,
            colsample_bytree=0.9,
            eval_metric='logloss',
            random_state=42,
            n_jobs=4
        ))
    ])

trained = {}
metrics_rows = []

for name, pipe in models.items():
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    # Some models need proba; fall back to decision function if needed
    if hasattr(pipe, 'predict_proba'):
        y_proba = pipe.predict_proba(X_test)[:, 1]
    elif hasattr(pipe, 'decision_function'):
        from sklearn.preprocessing import MinMaxScaler
        scores = pipe.decision_function(X_test).reshape(-1, 1)
        y_proba = MinMaxScaler().fit_transform(scores).ravel()
    else:
        # As a last resort, use predictions as probabilities
        y_proba = y_pred.astype(float)

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba)

    trained[name] = {
        'model': pipe,
        'y_pred': y_pred,
        'y_proba': y_proba,
    }

    metrics_rows.append({
        'model': name,
        'accuracy': acc,
        'precision': prec,
        'recall': rec,
        'f1': f1,
        'roc_auc': auc
    })

metrics_df = pd.DataFrame(metrics_rows).sort_values(by=['roc_auc','f1','accuracy'], ascending=False)
metrics_df


In [ ]:
# 5) Model Evaluation: Confusion Matrices and ROC Curves
from sklearn.metrics import RocCurveDisplay

plt.figure(figsize=(12,4))
for i, (name, obj) in enumerate(trained.items(), start=1):
    cm = confusion_matrix(y_test, obj['y_pred'])
    plt.subplot(1, len(trained), i)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False)
    plt.title(f'{name}\nConfusion Matrix')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'confusion_matrices.png'), dpi=150)
plt.show()

# ROC Curves
plt.figure(figsize=(6,5))
for name, obj in trained.items():
    fpr, tpr, _ = roc_curve(y_test, obj['y_proba'])
    auc = roc_auc_score(y_test, obj['y_proba'])
    plt.plot(fpr, tpr, label=f'{name} (AUC={auc:.3f})')
plt.plot([0,1],[0,1],'k--', alpha=0.5)
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves')
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'roc_curves.png'), dpi=150)
plt.show()

# Save metrics table
metrics_csv = os.path.join(REPORTS_DIR, 'model_metrics.csv')
metrics_df.to_csv(metrics_csv, index=False)
metrics_df


In [ ]:
# 6) Feature Importance & Interpretability
# Tree-based feature importance (RandomForest or XGBoost)
importances = None
feat_model_name = None
for name, obj in trained.items():
    mdl = obj['model']
    if hasattr(mdl, 'named_steps') and 'clf' in mdl.named_steps:
        clf = mdl.named_steps['clf']
    else:
        clf = mdl
    if hasattr(clf, 'feature_importances_'):
        importances = clf.feature_importances_
        feat_model_name = name
        break

if importances is not None:
    imp_df = pd.DataFrame({'feature': feature_names, 'importance': importances}).sort_values('importance', ascending=False)
    plt.figure(figsize=(8,6))
    sns.barplot(x='importance', y='feature', data=imp_df.head(15), palette='viridis')
    plt.title(f'Feature Importance ({feat_model_name}) - Top 15')
    plt.tight_layout()
    plt.savefig(os.path.join(FIGURES_DIR, 'feature_importance_top15.png'), dpi=150)
    plt.show()
    imp_df.head(15)
else:
    print('No tree-based model with feature_importances_ found.')

# SHAP interpretability (if available)
if HAS_SHAP:
    try:
        # Choose the best AUC model for SHAP
        best_name = metrics_df.iloc[0]['model']
        best_model = trained[best_name]['model']
        # Extract underlying estimator if wrapped in pipeline
        if hasattr(best_model, 'named_steps') and 'clf' in best_model.named_steps:
            estimator = best_model.named_steps['clf']
            # When a scaler is present, transform X for explanations
            X_explain = X_test.copy()
            if 'scaler' in best_model.named_steps:
                X_explain = pd.DataFrame(
                    best_model.named_steps['scaler'].transform(X_test),
                    columns=feature_names,
                    index=X_test.index
                )
        else:
            estimator = best_model
            X_explain = X_test

        if hasattr(estimator, 'predict_proba') or hasattr(estimator, 'decision_function'):
            if hasattr(estimator, 'predict_proba'):
                explainer = shap.Explainer(estimator.predict_proba, X_explain, feature_names=feature_names)
                shap_values = explainer(X_explain)
            else:
                explainer = shap.Explainer(estimator.decision_function, X_explain, feature_names=feature_names)
                shap_values = explainer(X_explain)
            plt.figure()
            shap.summary_plot(shap_values, X_explain, show=False)
            plt.tight_layout()
            plt.savefig(os.path.join(FIGURES_DIR, 'shap_summary.png'), dpi=150, bbox_inches='tight')
            plt.close()
            print('Saved SHAP summary plot.')
        else:
            print('Estimator does not support proba/decision_function for SHAP Explainer.')
    except Exception as e:
        print('SHAP failed:', e)
else:
    print('SHAP not available. Skipping.')


In [ ]:
# 7) Save best-performing model
best_row = metrics_df.iloc[0]
best_name = best_row['model']
best_model = trained[best_name]['model']

model_path = os.path.join(MODELS_DIR, f'best_model_{best_name}.joblib')
joblib.dump(best_model, model_path)
print('Best model saved to:', model_path)

# Also save the metrics to CSV (avoid optional tabulate dep)
metrics_csv = os.path.join(REPORTS_DIR, 'model_metrics.csv')
metrics_df.to_csv(metrics_csv, index=False)
print('Metrics CSV saved to:', metrics_csv)


## 8) Conclusion
- We trained Logistic Regression, Random Forest, and SVM (and XGBoost if available).
- We evaluated Accuracy, Precision, Recall, F1, and ROC-AUC; produced confusion matrices and ROC curves.
- We inspected feature importance from tree-based models and generated SHAP summary plots (if SHAP was available).
- The best model (by ROC-AUC) is saved under `models/` and metrics are saved in `reports/`.

Business insight: Features related to size and shape (e.g., `radius_worst`, `perimeter_worst`, `area_worst`, `concavity_worst`) typically show strong influence in predicting malignancy. A high AUC model with strong recall is preferable for screening to minimize false negatives.
